# Protein Structure Analysis with AlphaFold, Foldseek and ChimeraX

Welcome to this protein structure workshop! In this session, you will learn how to:
- Obtain protein structures
- Analyse sets of protein structures using Foldseek
- Analyse individual protein structures using ChimeraX

Each section includes explanations and small tasks or questions for you to work on.

---


## 1. How to Get Protein Structures

Protein structures can be obtained in two main ways:

### You can predict them yourself
- **Online**: Use [AlphaFold 3](https://alphafoldserver.com/welcome) to predict structures from amino acid sequences.
- **Offline**: Use AlphaFold locally on GPU clusters (e.g., LRZ).

### You can download existing ones from databases
- **Experimental**: RCSB PDB (https://www.rcsb.org)
- **Predicted**: AlphaFold DB (https://alphafold.ebi.ac.uk)

#### Task:
- Go to AlphaFold 3 (link above), have a look at the options for structure prediction and potentially try it out with an example.
- Go to AlphaFold DB and search for the keywords "estrogen receptor". Choose an entry: what files would you download for structure analysis?



## 2. How to Analyse a Set of Protein Structures (Foldseek)

### List of Estrogen Receptor Structures

This notebook uses a set of 5 receptor structures from different organisms, which are predicted structures from AlphaFold DB.

| Organism               | UniProt ID | Protein Name                     |
|------------------------|------------|----------------------------------|
| Human                 | P03372     | Estrogen receptor alpha         |
| Human                 | Q92731     | Estrogen receptor beta         |
| Mouse                 | P19785     | Estrogen receptor alpha         |
| Zebrafish             | P57717     | Estrogen receptor alpha         |
| Human                 | Q6UWB1     | Interleukin-27 receptor subunit alpha       |

Download URL format:
```
https://alphafold.ebi.ac.uk/files/AF-{UniProtID}-F1-model_v4.cif
```

## What can Foldseek do?

Foldseek allows fast and accurate comparison of protein structures. Input can be protein structures or amino acid sequences.

### Steps:
1. **Cluster** your set of structures by similarity.
2. **Create a database** of your protein structures.
3. **Search** a structure against the database.

### Note:
You can also use amino acid sequences as input if you have no structure available (or can also be of advantage if you use large datasets). For that, use **ProtT5** to generate 3Di sequences from your amino acid sequences. This is a 2D representative of your 3D structure and can be handled easily.

#### Questions:
- Which structure has the highest similarity score with `P03372`? What metrics help you determine this?
- Look at the list of files used: does the order of similarity make sense to you?
- Why is one structure not appearing? Can you change that?


**⚠️ Kernel:** Click **Select Kernel** (top right) → **Jupyter Kernel...** → **R (conda)**

In [1]:
BASE_PATH <- "/biodata/resources/protein_structure_analysis"

In [2]:
# Create output directories
system(paste("mkdir -p", file.path(BASE_PATH, "output_search")))
system(paste("mkdir -p", file.path(BASE_PATH, "output_cluster")))

In [3]:
# Cluster protein structures with Foldseek
system(paste("foldseek easy-cluster", file.path(BASE_PATH, "input"), file.path(BASE_PATH, "output_cluster/res_cluster"), file.path(BASE_PATH, "output_cluster/tmp_cluster"), "-c 0.9"))

In [4]:
# Load and display results
results_cluster <- read.table(file.path(BASE_PATH, 'output_cluster/res_cluster_cluster.tsv'), sep='\t', header=FALSE)
colnames(results_cluster) <- c('representative', 'member')
results_cluster

representative,member
<chr>,<chr>
AF-P57717-F1-model_v4,AF-P57717-F1-model_v4
AF-P57717-F1-model_v4,AF-Q92731-F1-model_v4
AF-P57717-F1-model_v4,AF-P19785-F1-model_v4
AF-P57717-F1-model_v4,AF-P03372-F1-model_v4
AF-Q6UWB1-F1-model_v4,AF-Q6UWB1-F1-model_v4


In [5]:
# Create database of your protein structures
system(paste("foldseek createdb", file.path(BASE_PATH, "input"), file.path(BASE_PATH, "output_search/est_rec_DB")))

In [14]:
# Search an example structrue against your database
system(paste("foldseek easy-search", file.path(BASE_PATH, "input/AF-P03372-F1-model_v4.cif"), file.path(BASE_PATH, "output_search/est_rec_DB"), file.path(BASE_PATH, "output_search/results"), file.path(BASE_PATH, "output_search/tmp"), '--format-output "query,target,pident,evalue,alnlen,alntmscore,qtmscore,lddt,prob,rmsd,bits"'))

In [15]:
# Load and display results
results_search <- read.table(file.path(BASE_PATH, 'output_search/results'), sep='\t', header=FALSE)
colnames(results_search) <- c('query', 'target', 'pident', 'evalue', 'alnlen', 'alntmscore', 'qtmscore', 'lddt', 'prob', 'rmsd', 'bits')
results_search

query,target,pident,evalue,alnlen,alntmscore,qtmscore,lddt,prob,rmsd,bits
<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
AF-P03372-F1-model_v4,AF-P03372-F1-model_v4,100.0,8.435e-80,595,1.0020,1.0000,1.0000,1,0.000,4030
AF-P03372-F1-model_v4,AF-P19785-F1-model_v4,88.9,5.828e-72,599,0.8094,0.8082,0.9581,1,6.498,3199
AF-P03372-F1-model_v4,AF-P57717-F1-model_v4,48.8,6.921e-44,588,0.5230,0.5004,0.8313,1,24.900,1465
AF-P03372-F1-model_v4,AF-Q92731-F1-model_v4,45.0,4.847e-41,582,0.6069,0.5438,0.8036,1,28.470,1451


In [ ]:
## DON'T RUN ##
# Option to get html output: --format-mode 3
foldseek easy-search $input_structure $path_to_est_rec_DB results.html tmp --format-mode 3 


## 3. How to Analyze Individual Protein Structures (ChimeraX)

ChimeraX is a powerful tool for visualizing and analyzing protein structures.

### Basic Usage:
- **Fetch structures** by PDB ID
- **Manipulate view**: rotate, zoom, select atoms/residues

### Interaction Analysis:
- Hydrogen bonds
- Van der Waals contacts

### Structure Comparison:
- RMSD calculation
- MatchMaker tool for alignment

#### Task:
- Visit the RCSB PDB and search for the PDB ID `1QKU`. 
What protein does this ID belong to? What organism is it from? What parts do you expect in the structure file?

- Open ChimeraX and fetch structure `1QKU`
- Identify the ligand bound to the receptor

Before you begin the analysis, do some small preparations:
-	Select chain A and B and the corresponding ligands (select one residue of the sequence, press **Up Arrow** keys until complete chain/ligand is selected) and write “del sel” in command line (= delete selection)
-	If command line not showing: > Tools > Command Line Interface
-	Select ER and hide all atoms (> Actions > Atoms/Bonds > Hide)
-	Select remaining ligand and color magenta (> Actions > Color)

To simplify the analysis, we look only at one chain (one part of homodimer) with its corresponding ligand!

Now you can start the analysis:
- Interaction & binding analysis:
    - Use Tools > Structure analysis > H bonds and Contacts to visualize interactions between the receptor and estradiol. Before you do this structure analysis, select the ligand. In the structure analysis panel, tick the option to show only contacts to the selection (with at least one end selected) and “Reveal atoms of H-bonding residues”.
    - Identify hydrogen bonds and Van der Waals contacts

To clean structure again for the next steps: hide bonds again [hide pseudobonds]
and select ER: > Actions > Atoms/Bonds > Hide 

- Structure comparison:
    - Load a second structure (e.g. mouse estrogen receptor 1VJB or human estrogen receptor gamma 1GPU)
    - Use Tools > Structure Analysis > Matchmaker to align the second structure to the first
    - Align the two structures using the command line: match #2 to #1
    - Check the RMSD value in the log panel

    **Bonus task**

    - Calculate RMSD for the binding region only by modifying the following expression "match #1/A:10-50 to #2/A:10-50"
    

#### Question:
- How many hydrogen bonds and Van der Waals contacts are formed between the receptor and estradiol?
- Which residues are involved in binding estradiol?

- What does RMSD tell you about two protein structures?
- What is the overall RMSD between the two structures?
- Which region(s) shows the most structural deviation?


